# Split estratificado y target en tres clases

Este notebook corresponde a la issue #12. Usa la función común `load_train_test()` para evitar que cada modelo repita la limpieza y la creación del target.

Los bins provisionales solo se usan para estratificar el split. Los umbrales definitivos de `Económico`, `Medio` y `Premium` se calculan exclusivamente con los precios de train.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path("..").resolve()))
from src.preprocessing import load_train_test

X_train, X_test, y_train, y_test, thresholds = load_train_test(
    return_thresholds=True
)
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"Umbral Económico/Medio: €{thresholds['lower']:.2f}")
print(f"Umbral Medio/Premium: €{thresholds['upper']:.2f}")

X_train: (10088, 71)
X_test: (2522, 71)
Umbral Económico/Medio: €122.00
Umbral Medio/Premium: €249.72


## Distribución de clases

In [2]:
distribution = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
})
distribution["train_pct"] = (distribution["train_count"] / len(y_train) * 100).round(2)
distribution["test_pct"] = (distribution["test_count"] / len(y_test) * 100).round(2)
display(distribution)

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert "price" not in X_train.columns
assert not set(X_train.index).intersection(X_test.index)

,train_count,test_count,train_pct,test_pct
price_segment,,,,
Económico,3366,841,33.37,33.35
Medio,3359,840,33.30,33.31
Premium,3363,841,33.34,33.35


## Decisiones

- El split es 80/20, estratificado y reproducible con `random_state=42`.
- Los bins provisionales equilibran el split, pero no son el target final.
- Los dos umbrales finales se calculan únicamente con `price_train`.
- Los mismos umbrales se aplican a `price_test`.
- La imputación, el escalado y el encoding quedan para #13 y se ajustarán solo con train.